<a href="https://colab.research.google.com/github/tungduong03/Deep-Learning/blob/main/secBERT-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Code_Injection_Dataset

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Code_Injection_Dataset


In [3]:
df = pd.read_csv('dataset_capec.csv')  # Đọc file CSV

In [4]:
# Optional (not effect very much)
df['text'] = df['text'].str.replace('/',' ')
df.head()

,text,label
0,GET blog index.php 2020 04 04 voluptatum-repr...,000 - Normal
1,GET blog xmlrpc.php?rsd,000 - Normal
2,GET blog index.php 2020 04 04 nihil-tenetur-e...,000 - Normal
3,GET blog index.php 2020 04 04 explicabo-qui-f...,000 - Normal
4,GET blog index.php 2020 04 04 explicabo-qui-f...,000 - Normal


In [5]:
# Đếm số lượng record cho mỗi loại label
label_counts = df['label'].value_counts()

# Lọc các label có số lượng record > 10000
labels_above_10000 = label_counts[label_counts > 10000].index

# Lấy 10,000 record đầu tiên cho các label đó
df_above_10000 = df[df['label'].isin(labels_above_10000)]
df_above_10000 = df_above_10000.groupby('label').head(10000)

# Lấy các record còn lại (cho các label không có số lượng lớn hơn 10000)
df_below_10000 = df[~df['label'].isin(labels_above_10000)]

# Ghép các record lại với nhau
df_combined = pd.concat([df_below_10000, df_above_10000])

# Xáo trộn dữ liệu sau khi ghép
df_combined = df_combined.sample(frac=1).reset_index(drop=True)

df = df_combined

In [6]:
# Phân chia dữ liệu train-test
X_train, X_test, y_train, y_test = train_test_split(df['text'],
                                                    df['label'],
                                                    test_size=0.2,
                                                    shuffle=True) # shuffle=True

In [16]:
# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)  # Encode train labels
y_test_encoded = label_encoder.transform(y_test)        # Encode test labels

# In danh sách nhãn đã mã hóa
print(label_encoder.classes_)  # ['000 - Normal', '001 - XSS', ...]
num_classes = len(label_encoder.classes_)  # Số lượng lớp
num_classes

['000 - Normal' '126 - Path Traversal' '153 - Input Data Manipulation'
 '194 - Fake the Source of Data' '242 - Code Injection'
 '272 - Protocol Manipulation' '310 - Scanning for Vulnerable Software'
 '34 - HTTP Response Splitting' '66 - SQL Injection']


9

In [12]:
# Tải Tokenizer của SecBERT
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("jackaduma/SecBERT")  # SecBERT

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
# Token hóa văn bản
# Tokenize dữ liệu train và test
def preprocess_data(texts, tokenizer, max_length=100):
    # Convert the Pandas Series to a list of strings
    texts = texts.tolist()  # This line is added to fix the error
    return tokenizer(
        texts,
        padding='max_length',        # Thêm padding
        truncation=True,             # Cắt ngắn văn bản
        max_length=max_length,       # Độ dài tối đa
        return_tensors='tf'          # Trả về Tensor
    )

X_train_tokens = preprocess_data(X_train, tokenizer)
X_test_tokens = preprocess_data(X_test, tokenizer)

In [13]:
from transformers import TFBertModel
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Model

# Tải mô hình SecBERT
secbert_model = TFBertModel.from_pretrained("jackaduma/SecBERT")


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'bert.embeddings.position_ids', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


In [11]:
from tensorflow.keras.layers import Lambda

# Input layers
input_ids = tf.keras.Input(shape=(100,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(100,), dtype=tf.int32, name="attention_mask")

# Gọi SecBERT qua Lambda layer, specify output_shape
bert_outputs = Lambda(
    lambda x: secbert_model(input_ids=x[0], attention_mask=x[1])[0][:, 0, :],
    output_shape=(768,)  # Specify the output shape here
)([input_ids, attention_mask])

# Thêm các lớp Dense
x = Dense(128, activation="relu")(bert_outputs)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation="softmax")(x)

# Tạo mô hình
model = Model(inputs=[input_ids, attention_mask], outputs=output)

# Compile mô hình
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [12]:
# huấn luyện mô hình
history = model.fit(
    {
        "input_ids": X_train_tokens["input_ids"],
        "attention_mask": X_train_tokens["attention_mask"]
    },
    y_train_encoded,
    validation_split=0.2,
    epochs=4,
    batch_size=128
)

Epoch 1/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 170s 443ms/step - accuracy: 0.3364 - loss: 1.7876 - val_accuracy: 0.4371 - val_loss: 1.4449
Epoch 2/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 195s 445ms/step - accuracy: 0.4157 - loss: 1.5203 - val_accuracy: 0.4464 - val_loss: 1.4211
Epoch 3/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 190s 410ms/step - accuracy: 0.4267 - loss: 1.4879 - val_accuracy: 0.4590 - val_loss: 1.4078
Epoch 4/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 215s 447ms/step - accuracy: 0.4324 - loss: 1.4764 - val_accuracy: 0.4627 - val_loss: 1.4024


In [13]:
test_loss, test_acc = model.evaluate(
    {"input_ids": X_test_tokens["input_ids"], "attention_mask": X_test_tokens["attention_mask"]},
    y_test_encoded
)
print(f"Test accuracy: {test_acc}")

442/442 ━━━━━━━━━━━━━━━━━━━━ 48s 99ms/step - accuracy: 0.4490 - loss: 1.4210
Test accuracy: 0.44684913754463196


**Bật fine-tune**

In [14]:
from tensorflow.keras.layers import Lambda

# Cho phép fine-tuning của mô hình SecBERT
secbert_model.trainable = True

# Input layers
input_ids = tf.keras.Input(shape=(100,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(100,), dtype=tf.int32, name="attention_mask")

# Gọi SecBERT qua Lambda layer, specify output_shape
bert_outputs = Lambda(
    lambda x: secbert_model(input_ids=x[0], attention_mask=x[1])[0][:, 0, :],
    output_shape=(768,)  # Specify the output shape here
)([input_ids, attention_mask])

# Thêm các lớp Dense
x = Dense(128, activation="relu")(bert_outputs)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation="softmax")(x)

# Tạo mô hình
model = Model(inputs=[input_ids, attention_mask], outputs=output)

# Compile mô hình
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [15]:
# huấn luyện mô hình
history = model.fit(
    {
        "input_ids": X_train_tokens["input_ids"],
        "attention_mask": X_train_tokens["attention_mask"]
    },
    y_train_encoded,
    validation_split=0.2,
    epochs=4,
    batch_size=128
)

Epoch 1/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 177s 472ms/step - accuracy: 0.3476 - loss: 1.7734 - val_accuracy: 0.4552 - val_loss: 1.4593
Epoch 2/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 189s 445ms/step - accuracy: 0.4091 - loss: 1.5289 - val_accuracy: 0.4430 - val_loss: 1.4300
Epoch 3/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 189s 409ms/step - accuracy: 0.4168 - loss: 1.4973 - val_accuracy: 0.4624 - val_loss: 1.4025
Epoch 4/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 202s 411ms/step - accuracy: 0.4244 - loss: 1.4905 - val_accuracy: 0.4523 - val_loss: 1.4075


In [16]:
test_loss, test_acc = model.evaluate(
    {"input_ids": X_test_tokens["input_ids"], "attention_mask": X_test_tokens["attention_mask"]},
    y_test_encoded
)
print(f"Test accuracy: {test_acc}")

442/442 ━━━━━━━━━━━━━━━━━━━━ 50s 100ms/step - accuracy: 0.4428 - loss: 1.4218
Test accuracy: 0.4387156069278717


**Giảm độ dài**

In [5]:
# Optional (not effect very much)
df['text'] = df['text'].str.replace('/',' ')
df.head()

,text,label
0,GET blog index.php 2020 04 04 voluptatum-repr...,000 - Normal
1,GET blog xmlrpc.php?rsd,000 - Normal
2,GET blog index.php 2020 04 04 nihil-tenetur-e...,000 - Normal
3,GET blog index.php 2020 04 04 explicabo-qui-f...,000 - Normal
4,GET blog index.php 2020 04 04 explicabo-qui-f...,000 - Normal


In [8]:
# Đếm số lượng record cho mỗi loại label
label_counts = df['label'].value_counts()

# Lọc các label có số lượng record > 10000
labels_above_10000 = label_counts[label_counts > 10000].index

# Lấy 10,000 record đầu tiên cho các label đó
df_above_10000 = df[df['label'].isin(labels_above_10000)]
df_above_10000 = df_above_10000.groupby('label').head(10000)

# Lấy các record còn lại (cho các label không có số lượng lớn hơn 10000)
df_below_10000 = df[~df['label'].isin(labels_above_10000)]

# Ghép các record lại với nhau
df_combined = pd.concat([df_below_10000, df_above_10000])

# Xáo trộn dữ liệu sau khi ghép
df_combined = df_combined.sample(frac=1).reset_index(drop=True)

df = df_combined

In [9]:
label_counts_df = label_counts.reset_index()
label_counts_df.columns = ['Label', 'Count']
print(label_counts_df)

                                    Label  Count
0                    242 - Code Injection  10000
1                    126 - Path Traversal  10000
2            34 - HTTP Response Splitting  10000
3           194 - Fake the Source of Data  10000
4                      66 - SQL Injection  10000
5                            000 - Normal  10000
6             272 - Protocol Manipulation   6924
7  310 - Scanning for Vulnerable Software   2382
8           153 - Input Data Manipulation   1387


In [10]:
# Phân chia dữ liệu train-test
X_train, X_test, y_train, y_test = train_test_split(df['text'],
                                                    df['label'],
                                                    test_size=0.2,
                                                    shuffle=True) # shuffle=True

In [14]:
# Token hóa văn bản
# Tokenize dữ liệu train và test
def preprocess_data(texts, tokenizer, max_length=95):
    # Convert the Pandas Series to a list of strings
    texts = texts.tolist()  # This line is added to fix the error
    return tokenizer(
        texts,
        padding='max_length',        # Thêm padding
        truncation=True,             # Cắt ngắn văn bản
        max_length=max_length,       # Độ dài tối đa
        return_tensors='tf'          # Trả về Tensor
    )

X_train_tokens = preprocess_data(X_train, tokenizer)
X_test_tokens = preprocess_data(X_test, tokenizer)

**No Fine-tune**

In [18]:
from tensorflow.keras.layers import Lambda

# Cho phép fine-tuning của mô hình SecBERT
secbert_model.trainable = False

# Input layers
input_ids = tf.keras.Input(shape=(95,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(95,), dtype=tf.int32, name="attention_mask")

# Gọi SecBERT qua Lambda layer, specify output_shape
bert_outputs = Lambda(
    lambda x: secbert_model(input_ids=x[0], attention_mask=x[1])[0][:, 0, :],
    output_shape=(768,)  # Specify the output shape here
)([input_ids, attention_mask])

# Thêm các lớp Dense
x = Dense(128, activation="relu")(bert_outputs)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation="softmax")(x)

# Tạo mô hình
model = Model(inputs=[input_ids, attention_mask], outputs=output)

# Compile mô hình
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [19]:
# huấn luyện mô hình
history = model.fit(
    {
        "input_ids": X_train_tokens["input_ids"],
        "attention_mask": X_train_tokens["attention_mask"]
    },
    y_train_encoded,
    validation_split=0.2,
    epochs=4,
    batch_size=128
)

Epoch 1/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 183s 475ms/step - accuracy: 0.3504 - loss: 1.7478 - val_accuracy: 0.4436 - val_loss: 1.4552
Epoch 2/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 188s 456ms/step - accuracy: 0.4180 - loss: 1.5246 - val_accuracy: 0.4456 - val_loss: 1.4216
Epoch 3/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 202s 455ms/step - accuracy: 0.4270 - loss: 1.4874 - val_accuracy: 0.4513 - val_loss: 1.4207
Epoch 4/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 191s 424ms/step - accuracy: 0.4299 - loss: 1.4774 - val_accuracy: 0.4504 - val_loss: 1.4237


**Tăng độ dài**

In [20]:
# Token hóa văn bản
# Tokenize dữ liệu train và test
def preprocess_data(texts, tokenizer, max_length=200):
    # Convert the Pandas Series to a list of strings
    texts = texts.tolist()  # This line is added to fix the error
    return tokenizer(
        texts,
        padding='max_length',        # Thêm padding
        truncation=True,             # Cắt ngắn văn bản
        max_length=max_length,       # Độ dài tối đa
        return_tensors='tf'          # Trả về Tensor
    )

X_train_tokens = preprocess_data(X_train, tokenizer)
X_test_tokens = preprocess_data(X_test, tokenizer)

In [21]:
from tensorflow.keras.layers import Lambda

# Cho phép fine-tuning của mô hình SecBERT
secbert_model.trainable = False

# Input layers
input_ids = tf.keras.Input(shape=(200,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(200,), dtype=tf.int32, name="attention_mask")

# Gọi SecBERT qua Lambda layer, specify output_shape
bert_outputs = Lambda(
    lambda x: secbert_model(input_ids=x[0], attention_mask=x[1])[0][:, 0, :],
    output_shape=(768,)  # Specify the output shape here
)([input_ids, attention_mask])

# Thêm các lớp Dense
x = Dense(128, activation="relu")(bert_outputs)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation="softmax")(x)

# Tạo mô hình
model = Model(inputs=[input_ids, attention_mask], outputs=output)

# Compile mô hình
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [22]:
# huấn luyện mô hình
history = model.fit(
    {
        "input_ids": X_train_tokens["input_ids"],
        "attention_mask": X_train_tokens["attention_mask"]
    },
    y_train_encoded,
    validation_split=0.2,
    epochs=4,
    batch_size=128
)

Epoch 1/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 339s 901ms/step - accuracy: 0.3377 - loss: 1.7992 - val_accuracy: 0.4420 - val_loss: 1.4583
Epoch 2/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 356s 856ms/step - accuracy: 0.4046 - loss: 1.5267 - val_accuracy: 0.4414 - val_loss: 1.4342
Epoch 3/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 322s 856ms/step - accuracy: 0.4161 - loss: 1.5031 - val_accuracy: 0.4495 - val_loss: 1.4186
Epoch 4/4
354/354 ━━━━━━━━━━━━━━━━━━━━ 343s 916ms/step - accuracy: 0.4229 - loss: 1.4886 - val_accuracy: 0.4413 - val_loss: 1.4162


In [23]:
# huấn luyện mô hình
history = model.fit(
    {
        "input_ids": X_train_tokens["input_ids"],
        "attention_mask": X_train_tokens["attention_mask"]
    },
    y_train_encoded,
    validation_split=0.2,
    epochs=4,
    batch_size=32
)

Epoch 1/4
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 346s 241ms/step - accuracy: 0.3981 - loss: 1.5398 - val_accuracy: 0.4333 - val_loss: 1.4358
Epoch 2/4
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 371s 237ms/step - accuracy: 0.4036 - loss: 1.5220 - val_accuracy: 0.4303 - val_loss: 1.4400
Epoch 3/4
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 363s 223ms/step - accuracy: 0.4020 - loss: 1.5123 - val_accuracy: 0.4309 - val_loss: 1.4362
Epoch 4/4
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 321s 223ms/step - accuracy: 0.4038 - loss: 1.5100 - val_accuracy: 0.4368 - val_loss: 1.4160


In [24]:
for i, layer in enumerate(secbert_model.layers):
    print(f"Layer {i}: {layer.name}, Trainable: {layer.trainable}")


Layer 0: bert, Trainable: False


In [27]:
from tensorflow.keras.layers import Lambda

# Cho phép fine-tuning của mô hình SecBERT
secbert_model.trainable = True

# Input layers
input_ids = tf.keras.Input(shape=(200,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(200,), dtype=tf.int32, name="attention_mask")

# Gọi SecBERT qua Lambda layer, specify output_shape
bert_outputs = Lambda(
    lambda x: secbert_model(input_ids=x[0], attention_mask=x[1])[0][:, 0, :],
    output_shape=(768,)  # Specify the output shape here
)([input_ids, attention_mask])

# Thêm các lớp Dense
x = Dense(128, activation="relu")(bert_outputs)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation="softmax")(x)

# Tạo mô hình
model = Model(inputs=[input_ids, attention_mask], outputs=output)

# Compile mô hình
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [28]:
for i, layer in enumerate(secbert_model.layers):
    print(f"Layer {i}: {layer.name}, Trainable: {layer.trainable}")


Layer 0: bert, Trainable: True


In [29]:
# huấn luyện mô hình
history = model.fit(
    {
        "input_ids": X_train_tokens["input_ids"],
        "attention_mask": X_train_tokens["attention_mask"]
    },
    y_train_encoded,
    validation_split=0.2,
    epochs=4,
    batch_size=32
)

Epoch 1/4
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 358s 240ms/step - accuracy: 0.3402 - loss: 1.7363 - val_accuracy: 0.4322 - val_loss: 1.4669
Epoch 2/4
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 368s 236ms/step - accuracy: 0.4004 - loss: 1.5449 - val_accuracy: 0.4375 - val_loss: 1.4429
Epoch 3/4
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 383s 237ms/step - accuracy: 0.4020 - loss: 1.5280 - val_accuracy: 0.4385 - val_loss: 1.4316
Epoch 4/4
1414/1414 ━━━━━━━━━━━━━━━━━━━━ 382s 237ms/step - accuracy: 0.4055 - loss: 1.5192 - val_accuracy: 0.4411 - val_loss: 1.4244


In [33]:
from tensorflow.keras.layers import Lambda

# Bật fine-tune
secbert_model.trainable = True

# Input layers
input_ids = tf.keras.Input(shape=(200,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(200,), dtype=tf.int32, name="attention_mask")

# Gọi SecBERT qua Lambda layer, specify output_shape
bert_outputs = Lambda(
    lambda x: secbert_model(input_ids=x[0], attention_mask=x[1])[0][:, 0, :],
    output_shape=(768,)  # Specify the output shape here
)([input_ids, attention_mask])

# Thêm các lớp Dense
x = Dense(256, activation="relu")(bert_outputs)
x = Dropout(0.25)(x)
output = Dense(num_classes, activation="softmax")(x)

# Tạo mô hình
model = Model(inputs=[input_ids, attention_mask], outputs=output)

# Compile mô hình
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [34]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    {
        "input_ids": X_train_tokens["input_ids"],
        "attention_mask": X_train_tokens["attention_mask"]
    },
    y_train_encoded,
    validation_split=0.2,
    epochs=15,
    batch_size=128,
    callbacks=[early_stopping]
)


Epoch 1/15
354/354 ━━━━━━━━━━━━━━━━━━━━ 327s 892ms/step - accuracy: 0.1793 - loss: 2.2421 - val_accuracy: 0.3773 - val_loss: 1.7582
Epoch 2/15
354/354 ━━━━━━━━━━━━━━━━━━━━ 303s 853ms/step - accuracy: 0.3228 - loss: 1.8349 - val_accuracy: 0.4077 - val_loss: 1.6321
Epoch 3/15
354/354 ━━━━━━━━━━━━━━━━━━━━ 302s 853ms/step - accuracy: 0.3789 - loss: 1.6765 - val_accuracy: 0.4317 - val_loss: 1.5731
Epoch 4/15
354/354 ━━━━━━━━━━━━━━━━━━━━ 323s 855ms/step - accuracy: 0.3963 - loss: 1.6220 - val_accuracy: 0.4403 - val_loss: 1.5388
Epoch 5/15
354/354 ━━━━━━━━━━━━━━━━━━━━ 321s 854ms/step - accuracy: 0.4044 - loss: 1.5859 - val_accuracy: 0.4408 - val_loss: 1.5163
Epoch 6/15
354/354 ━━━━━━━━━━━━━━━━━━━━ 344s 915ms/step - accuracy: 0.4188 - loss: 1.5549 - val_accuracy: 0.4413 - val_loss: 1.4969
Epoch 7/15
354/354 ━━━━━━━━━━━━━━━━━━━━ 361s 855ms/step - accuracy: 0.4214 - loss: 1.5384 - val_accuracy: 0.4426 - val_loss: 1.4836
Epoch 8/15
354/354 ━━━━━━━━━━━━━━━━━━━━ 344s 917ms/step - accuracy: 0.4205 -

In [35]:
test_loss, test_acc = model.evaluate(
    {"input_ids": X_test_tokens["input_ids"], "attention_mask": X_test_tokens["attention_mask"]},
    y_test_encoded
)
print(f"Test accuracy: {test_acc}")

442/442 ━━━━━━━━━━━━━━━━━━━━ 86s 187ms/step - accuracy: 0.4537 - loss: 1.4278
Test accuracy: 0.45844826102256775


In [36]:
from sklearn.metrics import classification_report

# Dự đoán trên tập validation hoặc test
y_pred = model.predict({
    "input_ids": X_test_tokens["input_ids"],
    "attention_mask": X_test_tokens["attention_mask"]
})

# Lấy nhãn dự đoán từ xác suất (với softmax)
y_pred_labels = y_pred.argmax(axis=1)

label_names = label_encoder.classes_  # Lấy danh sách tên nhãn

# Tính toán precision, recall, f1-score
print(classification_report(y_test_encoded, y_pred_labels, target_names=label_names))

442/442 ━━━━━━━━━━━━━━━━━━━━ 86s 188ms/step
                                        precision    recall  f1-score   support

                          000 - Normal       0.34      0.52      0.41      2044
                  126 - Path Traversal       0.79      0.74      0.77      1995
         153 - Input Data Manipulation       0.29      0.05      0.09       238
         194 - Fake the Source of Data       0.43      0.32      0.37      1983
                  242 - Code Injection       0.50      0.68      0.58      1980
           272 - Protocol Manipulation       0.35      0.38      0.37      1394
310 - Scanning for Vulnerable Software       0.66      0.58      0.62       502
          34 - HTTP Response Splitting       0.35      0.25      0.29      1984
                    66 - SQL Injection       0.41      0.31      0.36      2019

                              accuracy                           0.46     14139
                             macro avg       0.46      0.43      0.43     